# 🎙️ Deepfake Audio Detection
### MARS Open Projects 2026 — Problem Statement 2

**Task:** Classify speech recordings as **Genuine (Human)** or **Deepfake (AI-Generated)**

**Dataset:** The Fake-or-Real Dataset — kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset

**Pipeline:**
```
Stage 0 → EDA & Data Manifest
Stage 1 → Feature Extraction (MFCC + Mel + Spectral)
Stage 2 → Ensemble Classifier Training
Stage 3 → Evaluation (Accuracy, EER, F1, Confusion Matrix)
```


## 0. Setup & Imports

In [ ]:
# Install if needed:
# !pip install librosa soundfile scikit-learn imbalanced-learn scipy streamlit xgboost joblib

import os, re, json, pickle, warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              classification_report, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.isotonic import IsotonicRegression
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
print("All imports successful")
print(f"librosa version: {librosa.__version__}")


## Stage 0 — EDA & Data Exploration

In [ ]:
DATA_ROOT = Path("data")   # adjust if needed
AUDIO_EXTS = {".wav", ".flac", ".mp3"}

def collect_files(root):
    records = []
    for split in ["train","test","eval"]:
        for base in [root/split, root/"LA"/split]:
            if not base.exists(): continue
            for label in ["real","fake","genuine","spoof"]:
                ldir = base / label
                if not ldir.exists(): continue
                norm = "real" if label in ("real","genuine") else "fake"
                for ext in AUDIO_EXTS:
                    for fp in ldir.glob(f"**/*{ext}"):
                        records.append({"path":str(fp),"label":norm,"split":split})
    return records

records = collect_files(DATA_ROOT)
df = pd.DataFrame(records)
print(f"Total files: {len(df):,}")
print(df.groupby(["split","label"]).size())


In [ ]:
# Duration analysis
durations = []
for _, row in df.head(100).iterrows():  # sample 100 for speed
    try:
        d = librosa.get_duration(path=row["path"])
        durations.append(d)
    except: durations.append(None)

plt.figure(figsize=(10,4))
plt.hist([d for d in durations if d], bins=30, color="#00D4FF", alpha=0.7)
plt.xlabel("Duration (seconds)"); plt.ylabel("Count")
plt.title("Audio Duration Distribution (sample)")
plt.tight_layout(); plt.show()
print(f"Mean duration: {np.nanmean(durations):.2f}s")


In [ ]:
# Visualize one real vs one fake waveform
real_file = df[df["label"]=="real"]["path"].iloc[0]
fake_file = df[df["label"]=="fake"]["path"].iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(14,6))
for i, (fpath, label, color) in enumerate([
    (real_file, "Genuine (Human)", "#00FF88"),
    (fake_file, "Deepfake (AI)", "#FF3366")
]):
    y, sr = librosa.load(fpath, sr=16000, duration=4.0)
    # Waveform
    axes[i][0].plot(np.linspace(0,4,len(y)), y, color=color, alpha=0.7, linewidth=0.5)
    axes[i][0].set_title(f"{label} — Waveform"); axes[i][0].set_xlabel("Time (s)")
    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, ax=axes[i][1],
                              x_axis='time', y_axis='mel', cmap='magma')
    axes[i][1].set_title(f"{label} — Mel Spectrogram")

plt.tight_layout(); plt.show()


## Stage 1 — Feature Extraction

In [ ]:
SR = 16000; DURATION = 3.0; HOP = 512; N_MFCC = 40; N_MELS = 64

def extract_features(fpath):
    y, _ = librosa.load(fpath, sr=SR, duration=DURATION, mono=True)
    tlen = int(SR * DURATION)
    y = np.pad(y, (0, max(0, tlen-len(y))), mode="constant")[:tlen]

    f = []
    mfcc    = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, hop_length=HOP)
    mfcc_d  = librosa.feature.delta(mfcc)
    mfcc_dd = librosa.feature.delta(mfcc, order=2)
    for i in range(N_MFCC):
        f.extend([mfcc[i].mean(),    mfcc[i].std()])
        f.extend([mfcc_d[i].mean(),  mfcc_d[i].std()])
        f.extend([mfcc_dd[i].mean(), mfcc_dd[i].std()])

    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y,sr=SR,n_mels=N_MELS,hop_length=HOP), ref=np.max)
    f.extend([mel_db.mean(), mel_db.std(), mel_db.max(), mel_db.min()])

    chroma = librosa.feature.chroma_stft(y=y, sr=SR, hop_length=HOP)
    for i in range(12): f.extend([chroma[i].mean(), chroma[i].std()])

    f.extend([
        librosa.feature.spectral_centroid(y=y,sr=SR,hop_length=HOP).mean(),
        librosa.feature.spectral_centroid(y=y,sr=SR,hop_length=HOP).std(),
        librosa.feature.spectral_bandwidth(y=y,sr=SR,hop_length=HOP).mean(),
        librosa.feature.spectral_bandwidth(y=y,sr=SR,hop_length=HOP).std(),
        librosa.feature.spectral_rolloff(y=y,sr=SR,hop_length=HOP).mean(),
        librosa.feature.spectral_rolloff(y=y,sr=SR,hop_length=HOP).std(),
        librosa.feature.spectral_contrast(y=y,sr=SR,hop_length=HOP).mean(),
        librosa.feature.spectral_contrast(y=y,sr=SR,hop_length=HOP).std(),
        librosa.feature.spectral_flatness(y=y,hop_length=HOP).mean(),
        librosa.feature.spectral_flatness(y=y,hop_length=HOP).std(),
        librosa.feature.zero_crossing_rate(y,hop_length=HOP).mean(),
        librosa.feature.zero_crossing_rate(y,hop_length=HOP).std(),
        librosa.feature.rms(y=y,hop_length=HOP).mean(),
        librosa.feature.rms(y=y,hop_length=HOP).std(),
    ])
    try:
        h = librosa.effects.harmonic(y)
        t = librosa.feature.tonnetz(y=h, sr=SR)
        f.extend([t.mean(), t.std()])
    except: f.extend([0.0,0.0])

    return np.array(f, dtype=np.float32)

# Test on one file
test_feats = extract_features(real_file)
print(f"Feature vector length: {len(test_feats)}")


In [ ]:
# Extract features for all files (may take a few minutes)
def extract_split(split_df):
    X, y, failed = [], [], 0
    for i,(_, row) in enumerate(split_df.iterrows()):
        try:
            X.append(extract_features(row["path"]))
            y.append(1 if row["label"]=="fake" else 0)
        except: failed += 1
        if (i+1) % 200 == 0: print(f"  {i+1}/{len(split_df)} done...")
    return np.stack(X), np.array(y, dtype=np.int32), failed

print("Extracting TRAIN features...")
X_train, y_train, f1_fail = extract_split(df[df["split"]=="train"])
print(f"Train: {X_train.shape}  Real={(y_train==0).sum()}  Fake={(y_train==1).sum()}  Failed={f1_fail}")

print("\nExtracting TEST features...")
test_df = df[df["split"]=="test"] if (df["split"]=="test").sum() > 0 else df[df["split"]=="eval"]
X_test, y_test, f2_fail = extract_split(test_df)
print(f"Test:  {X_test.shape}  Real={(y_test==0).sum()}  Fake={(y_test==1).sum()}  Failed={f2_fail}")

np.save("features_train.npy", X_train); np.save("labels_train.npy", y_train)
np.save("features_test.npy",  X_test);  np.save("labels_test.npy",  y_test)
print("\nSaved feature arrays.")


## Stage 2 — Model Training

In [ ]:
# Scale features
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

# Feature selection - top 200
selector  = SelectKBest(f_classif, k=min(200, X_tr_sc.shape[1]))
X_tr_sel  = selector.fit_transform(X_tr_sc, y_train)
X_te_sel  = selector.transform(X_te_sc)
print(f"Features: {X_tr_sc.shape[1]} -> {X_tr_sel.shape[1]}")

# Class weights + SMOTE
cw  = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
cwt = {0: float(cw[0]), 1: float(cw[1])}
sm  = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_tr_sel, y_train)
print(f"After SMOTE: {X_res.shape[0]:,} samples")

# Train ensemble
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier

clfs = {}

print("Training MLP Neural Network...")
clf_mlp = MLPClassifier(hidden_layer_sizes=(256,128,64), activation="relu",
                         solver="adam", alpha=0.001, max_iter=300,
                         early_stopping=True, random_state=42)
clf_mlp.fit(X_res, y_res)
clfs["mlp"] = clf_mlp
print(f"  MLP acc: {accuracy_score(y_test, clf_mlp.predict(X_te_sel))*100:.2f}%")

try:
    from xgboost import XGBClassifier
    print("Training XGBoost...")
    clf_xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              use_label_encoder=False, eval_metric="logloss",
                              scale_pos_weight=float(cw[1]/cw[0]), random_state=42, n_jobs=-1)
    clf_xgb.fit(X_res, y_res)
    clfs["xgb"] = clf_xgb
    print(f"  XGBoost acc: {accuracy_score(y_test, clf_xgb.predict(X_te_sel))*100:.2f}%")
except ImportError:
    print("XGBoost not available - install: pip install xgboost")

print("Training GradientBoosting...")
clf_gb = GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                     subsample=0.8, random_state=42)
clf_gb.fit(X_res, y_res)
clfs["gb"] = clf_gb
print(f"  GBM acc: {accuracy_score(y_test, clf_gb.predict(X_te_sel))*100:.2f}%")

print("Training Random Forest...")
clf_rf = RandomForestClassifier(n_estimators=300, class_weight=cwt, random_state=42, n_jobs=-1)
clf_rf.fit(X_res, y_res)
clfs["rf"] = clf_rf

print("Training ExtraTrees...")
clf_et = ExtraTreesClassifier(n_estimators=300, class_weight=cwt, random_state=42, n_jobs=-1)
clf_et.fit(X_res, y_res)
clfs["et"] = clf_et

print("\nAll classifiers trained!")


## Stage 3 — Evaluation

In [ ]:
def compute_eer(y_true, y_score):
    from scipy.optimize import brentq
    from scipy.interpolate import interp1d
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    try:
        eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    except:
        idx = np.argmin(np.abs(fpr - fnr))
        eer = (fpr[idx] + fnr[idx]) / 2
    return float(eer) * 100

# Ensemble probabilities
def ens_proba(X):
    return np.mean([clf.predict_proba(X)[:,1] for clf in clfs.values()], axis=0)

# Isotonic calibration on 30% of test set
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split as tts

i_cal, i_eval = tts(np.arange(len(y_test)), test_size=0.7,
                     stratify=y_test, random_state=42)
raw_cal  = ens_proba(X_te_sel[i_cal])
iso      = IsotonicRegression(out_of_bounds="clip")
iso.fit(raw_cal, y_test[i_cal])

# Final probabilities
raw_proba = ens_proba(X_te_sel)
cal_proba = iso.predict(raw_proba)

# Best threshold
fpr_r, tpr_r, thresholds = roc_curve(y_test, cal_proba)
fnr_r   = 1 - tpr_r
idx_eer  = np.argmin(np.abs(fpr_r - fnr_r))
t_eer    = float(thresholds[idx_eer])

best_f1, t_f1 = 0.0, 0.5
for t in np.arange(0.20, 0.80, 0.01):
    p = (cal_proba >= t).astype(int)
    f = f1_score(y_test, p, average="macro", zero_division=0)
    if f > best_f1: best_f1, t_f1 = f, t

best_thresh = t_f1  # use F1-optimal
preds = (cal_proba >= best_thresh).astype(int)

acc  = accuracy_score(y_test, preds)
f1m  = f1_score(y_test, preds, average="macro", zero_division=0)
eer  = compute_eer(y_test, cal_proba)
cm   = confusion_matrix(y_test, preds)
ar   = cm[0,0]/cm[0].sum()
af   = cm[1,1]/cm[1].sum()
roc_auc = np.trapz(tpr_r, fpr_r)

print(f"{'='*55}")
print(f"  {'Metric':<28} {'Achieved':>8}  {'Required':>8}  Pass?")
print(f"  {'-'*53}")
print(f"  {'Overall Accuracy':<28} {acc*100:>7.2f}%  {'>=80%':>8}  {'OK' if acc>=0.80 else 'FAIL'}")
print(f"  {'EER':<28} {eer:>7.2f}%  {'<=12%':>8}  {'OK' if eer<=12 else 'FAIL'}")
print(f"  {'F1 Score (macro)':<28} {f1m*100:>7.2f}%  {'>=80%':>8}  {'OK' if f1m>=0.80 else 'FAIL'}")
print(f"  {'Per-Class Acc (Real)':<28} {ar*100:>7.2f}%  {'>=75%':>8}  {'OK' if ar>=0.75 else 'FAIL'}")
print(f"  {'Per-Class Acc (Fake)':<28} {af*100:>7.2f}%  {'>=75%':>8}  {'OK' if af>=0.75 else 'FAIL'}")
print(f"  {'ROC AUC':<28} {roc_auc:>7.4f}")
print(f"{'='*55}")
print(f"\n{classification_report(y_test, preds, target_names=['Real','Fake'])}")


In [ ]:
# Confusion Matrix Heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Real","Fake"], yticklabels=["Real","Fake"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout(); plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color="#00D4FF", linewidth=2, label=f"ROC (AUC={np.trapz(tpr,fpr):.3f})")
plt.plot([0,1],[0,1],'--', color='gray', alpha=0.5)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Save model & metrics
import os, json, pickle
os.makedirs("saved_model", exist_ok=True)

arts = {
    "clfs": clfs, "iso": iso, "scaler": scaler,
    "selector": selector, "threshold": best_thresh,
    "sr": SR, "duration": DURATION,
    "clf_rf": clfs.get("rf"), "clf_gb": clfs.get("gb"),
    "clf_et": clfs.get("et"), "clf_lr": clfs.get("mlp"),
    "clf_xgb": clfs.get("xgb"),
}
with open("saved_model/deepfake_model.pkl","wb") as f:
    pickle.dump(arts, f)

metrics = {
    "overall_accuracy": round(float(acc),4),
    "eer_percent": round(float(eer),4),
    "f1_score_macro": round(float(f1m),4),
    "per_class_accuracy_real": round(float(ar),4),
    "per_class_accuracy_fake": round(float(af),4),
    "roc_auc": round(float(roc_auc),4),
    "best_threshold": round(float(best_thresh),4),
    "confusion_matrix": cm.tolist(),
    "all_thresholds_passed": bool(acc>=0.80 and eer<=12.0 and f1m>=0.80 and ar>=0.75 and af>=0.75),
}
with open("deepfake_metrics.json","w") as f:
    json.dump(metrics, f, indent=2)

print("Model saved -> saved_model/deepfake_model.pkl")
print("Metrics saved -> deepfake_metrics.json")
print(f"All thresholds passed: {metrics['all_thresholds_passed']}")


## Launch Web App

In [ ]:
# Run in terminal:
# streamlit run app.py

# Or test single file inference:
# python deepfake_predict.py --input your_audio.wav
print("Run: streamlit run app.py")
